# Kaggle Upload / Download

Notebook này có hai nhánh upload:

- `UPLOAD_TARGET=model`: stage `best` và `last` checkpoint từ một hoặc nhiều `run_name`, rồi upload lên **Kaggle Models**.
- `UPLOAD_TARGET=dataset`: stage thư mục `public/`, rồi upload lên **Kaggle Datasets**.
- `UPLOAD_TARGET=both`: chạy cả hai nhánh.

Cấu trúc Kaggle Models có 3 tầng:
- **Model**: ví dụ `ngocbaotrinhtuan/object-detection-checkpoints`
- **Model variation / instance**: ví dụ `ngocbaotrinhtuan/object-detection-checkpoints/PyTorch/checkpoints`
- **Version**: nơi chứa file checkpoint thực tế

`download.sh` hiện vẫn dùng cho flow tải/copy checkpoint local. Nếu muốn tải từ Kaggle Models, dùng lệnh ở cuối notebook.


In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "saved_results").exists() and (PROJECT_ROOT.parent / "saved_results").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent


def parse_run_names(value: str | None, default: list[str]) -> list[str]:
    if not value:
        return default
    value = value.strip()
    if not value:
        return default
    if value.startswith("["):
        parsed = json.loads(value)
        return [str(item).strip() for item in parsed if str(item).strip()]
    return [item.strip() for item in value.split(",") if item.strip()]


UPLOAD_TARGET = 'dataset'
if UPLOAD_TARGET not in {"model", "dataset", "both"}:
    raise ValueError("UPLOAD_TARGET must be one of: model, dataset, both")

RUN_NAMES = parse_run_names(
    os.getenv("RUN_NAMES"),
    ["custom-baseline", "torchvision-augmentmax", "retina-baseline"],
)

SAVED_RESULTS_ROOT = Path(os.getenv("SAVED_RESULTS_ROOT", PROJECT_ROOT / "saved_results"))
EXPORT_ROOT = Path(os.getenv("EXPORT_ROOT", SAVED_RESULTS_ROOT / "kaggle_upload_export"))
MODEL_METADATA_DIR = EXPORT_ROOT / "_model_metadata"
INSTANCE_METADATA_DIR = EXPORT_ROOT / "_instance_metadata"
MODEL_VERSION_DIR = EXPORT_ROOT / "model_version"
DATASET_EXPORT_DIR = EXPORT_ROOT / "public_dataset"

KAGGLE_OWNER = os.getenv("KAGGLE_OWNER", os.getenv("KAGGLE_USERNAME", "ngocbaotrinhtuan")).strip()
KAGGLE_MODEL_SLUG = os.getenv("KAGGLE_MODEL_SLUG", "object-detection-checkpoints").strip()
KAGGLE_MODEL_TITLE = os.getenv("KAGGLE_MODEL_TITLE", "Object Detection Checkpoints").strip()
KAGGLE_MODEL_SUBTITLE = os.getenv("KAGGLE_MODEL_SUBTITLE", "PyTorch checkpoints for object detection experiments").strip()
KAGGLE_MODEL_PRIVATE = os.getenv("KAGGLE_MODEL_PRIVATE", "1").strip().lower() not in {"0", "false", "no"}

KAGGLE_FRAMEWORK = os.getenv("KAGGLE_FRAMEWORK", "PyTorch").strip()
KAGGLE_INSTANCE_SLUG = os.getenv("KAGGLE_INSTANCE_SLUG", "checkpoints").strip()
KAGGLE_MODEL_LICENSE = os.getenv("KAGGLE_MODEL_LICENSE", "Apache 2.0").strip()
KAGGLE_MODEL_INSTANCE_REF = f"{KAGGLE_OWNER}/{KAGGLE_MODEL_SLUG}/{KAGGLE_FRAMEWORK}/{KAGGLE_INSTANCE_SLUG}"

KAGGLE_DATASET_SLUG = os.getenv("KAGGLE_DATASET_SLUG", f"{KAGGLE_OWNER}/object-detection-public").strip()
KAGGLE_DATASET_TITLE = os.getenv("KAGGLE_DATASET_TITLE", "Object Detection Public Dataset").strip()
KAGGLE_DATASET_LICENSE = os.getenv("KAGGLE_DATASET_LICENSE", "CC0-1.0").strip()
PUBLIC_DIR = Path(os.getenv("PUBLIC_DIR", PROJECT_ROOT / "public"))

CREATE_MODEL_IF_MISSING = os.getenv("CREATE_MODEL_IF_MISSING", "1").strip().lower() not in {"0", "false", "no"}
CREATE_INSTANCE_IF_MISSING = os.getenv("CREATE_INSTANCE_IF_MISSING", "1").strip().lower() not in {"0", "false", "no"}
VERSION_NOTES = os.getenv("VERSION_NOTES", f"Update checkpoints for {len(RUN_NAMES)} runs: {', '.join(RUN_NAMES)}")
DATASET_VERSION_NOTES = os.getenv("DATASET_VERSION_NOTES", "Update public object-detection dataset")

for directory in [EXPORT_ROOT, MODEL_METADATA_DIR, INSTANCE_METADATA_DIR, MODEL_VERSION_DIR, DATASET_EXPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "project_root": str(PROJECT_ROOT),
    "upload_target": UPLOAD_TARGET,
    "run_names": RUN_NAMES,
    "saved_results_root": str(SAVED_RESULTS_ROOT),
    "export_root": str(EXPORT_ROOT),
    "model_version_dir": str(MODEL_VERSION_DIR),
    "dataset_export_dir": str(DATASET_EXPORT_DIR),
    "public_dir": str(PUBLIC_DIR),
    "kaggle_model": f"{KAGGLE_OWNER}/{KAGGLE_MODEL_SLUG}",
    "kaggle_model_instance": KAGGLE_MODEL_INSTANCE_REF,
    "private_model": KAGGLE_MODEL_PRIVATE,
    "kaggle_dataset": KAGGLE_DATASET_SLUG,
}, indent=2, ensure_ascii=False))


{
  "project_root": "/Users/ngocbao/Documents/Document/Semester6/Image Processing/Final",
  "upload_target": "dataset",
  "run_names": [
    "custom-baseline",
    "torchvision-augmentmax",
    "retina-baseline"
  ],
  "saved_results_root": "/Users/ngocbao/Documents/Document/Semester6/Image Processing/Final/saved_results",
  "export_root": "/Users/ngocbao/Documents/Document/Semester6/Image Processing/Final/saved_results/kaggle_upload_export",
  "model_version_dir": "/Users/ngocbao/Documents/Document/Semester6/Image Processing/Final/saved_results/kaggle_upload_export/model_version",
  "dataset_export_dir": "/Users/ngocbao/Documents/Document/Semester6/Image Processing/Final/saved_results/kaggle_upload_export/public_dataset",
  "public_dir": "public",
  "kaggle_model": "ngocbaotrinhtuan/object-detection-checkpoints",
  "kaggle_model_instance": "ngocbaotrinhtuan/object-detection-checkpoints/PyTorch/checkpoints",
  "private_model": true,
  "kaggle_dataset": "ngocbaotrinhtuan/object-detectio

In [2]:
def latest_checkpoint(checkpoint_dir: Path, pattern: str, fallback_name: str) -> Path | None:
    matches = sorted(checkpoint_dir.glob(pattern), key=lambda path: path.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    fallback = checkpoint_dir / fallback_name
    return fallback if fallback.exists() else None


def stage_checkpoint(source: Path | None, destination: Path) -> Path | None:
    if source is None:
        return None
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    return destination


def stage_model_checkpoints() -> list[dict[str, str | None]]:
    if MODEL_VERSION_DIR.exists():
        shutil.rmtree(MODEL_VERSION_DIR)
    MODEL_VERSION_DIR.mkdir(parents=True, exist_ok=True)

    staged_runs = []
    for run_name in RUN_NAMES:
        checkpoint_dir = SAVED_RESULTS_ROOT / run_name / "checkpoints"
        best_source = latest_checkpoint(checkpoint_dir, "best_model-*.pth", "best_model.pth")
        last_source = latest_checkpoint(checkpoint_dir, "last_model-*.pth", "last_model.pth")

        run_export_dir = MODEL_VERSION_DIR / run_name / "checkpoints"
        best_target = stage_checkpoint(best_source, run_export_dir / "best_model.pth")
        last_target = stage_checkpoint(last_source, run_export_dir / "last_model.pth")

        staged_runs.append({
            "run_name": run_name,
            "checkpoint_dir": str(checkpoint_dir),
            "best_source": str(best_source) if best_source else None,
            "last_source": str(last_source) if last_source else None,
            "best_target": str(best_target) if best_target else None,
            "last_target": str(last_target) if last_target else None,
        })

    manifest_path = MODEL_VERSION_DIR / "checkpoint-manifest.json"
    manifest_path.write_text(json.dumps(staged_runs, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

    print(json.dumps(staged_runs, indent=2, ensure_ascii=False))
    print(f"Manifest written: {manifest_path}")

    if not any(item["best_target"] or item["last_target"] for item in staged_runs):
        raise FileNotFoundError(f"No checkpoint found for all runs in {SAVED_RESULTS_ROOT}")
    return staged_runs


def write_dataset_metadata(export_dir: Path) -> Path:
    metadata = {
        "title": KAGGLE_DATASET_TITLE,
        "id": KAGGLE_DATASET_SLUG,
        "licenses": [{"name": KAGGLE_DATASET_LICENSE}],
    }
    path = export_dir / "dataset-metadata.json"
    path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path


def stage_public_dataset() -> Path:
    if not PUBLIC_DIR.exists():
        raise FileNotFoundError(f"Public dataset directory not found: {PUBLIC_DIR}")
    if DATASET_EXPORT_DIR.exists():
        shutil.rmtree(DATASET_EXPORT_DIR)
    DATASET_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    target_public_dir = DATASET_EXPORT_DIR / "public"
    ignore = shutil.ignore_patterns(".DS_Store", "__pycache__", "*.pyc")
    shutil.copytree(PUBLIC_DIR, target_public_dir, ignore=ignore)
    metadata_path = write_dataset_metadata(DATASET_EXPORT_DIR)

    total_files = sum(1 for path in target_public_dir.rglob("*") if path.is_file())
    print(json.dumps({
        "source": str(PUBLIC_DIR),
        "staged_public_dir": str(target_public_dir),
        "metadata": str(metadata_path),
        "num_files": total_files,
    }, indent=2, ensure_ascii=False))
    return DATASET_EXPORT_DIR


if UPLOAD_TARGET in {"model", "both"}:
    stage_model_checkpoints()
else:
    print("Skipping model checkpoint staging.")

if UPLOAD_TARGET in {"dataset", "both"}:
    stage_public_dataset()
else:
    print("Skipping public dataset staging.")


Skipping model checkpoint staging.


{
  "source": "public",
  "staged_public_dir": "/Users/ngocbao/Documents/Document/Semester6/Image Processing/Final/saved_results/kaggle_upload_export/public_dataset/public",
  "metadata": "/Users/ngocbao/Documents/Document/Semester6/Image Processing/Final/saved_results/kaggle_upload_export/public_dataset/dataset-metadata.json",
  "num_files": 9006
}


In [3]:
def write_model_metadata(output_dir: Path) -> Path:
    description = f"""# Model Summary

Object detection checkpoint collection for this project.

# Model Characteristics

This Kaggle Model stores PyTorch checkpoint files exported from `saved_results/<run_name>/checkpoints`.

# Data Overview

Dataset is managed separately from model checkpoints.

# Evaluation Results

See each run folder and local `saved_results` logs/evaluation files for metrics.
"""
    metadata = {
        "ownerSlug": KAGGLE_OWNER,
        "title": KAGGLE_MODEL_TITLE,
        "slug": KAGGLE_MODEL_SLUG,
        "subtitle": KAGGLE_MODEL_SUBTITLE,
        "isPrivate": KAGGLE_MODEL_PRIVATE,
        "description": description,
        "publishTime": "",
        "provenanceSources": "",
    }
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / "model-metadata.json"
    path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path


def write_model_instance_metadata(output_dir: Path) -> Path:
    usage = f"""# Model Format

PyTorch `.pth` checkpoints are stored under `<run_name>/checkpoints/`.

# Training Data

Training data is not bundled in this model artifact.

# Model Inputs

Use the repository `predict.py` script with the matching run configuration.

# Model Outputs

The model produces object-detection predictions in JSON format.

# Model Usage

Download this Kaggle Model variation version, then place checkpoints under `saved_results/<run_name>/checkpoints/`.

# Fine-tuning

Use `train.py` or `script.sh train` with `--resume_from` / `RESUME_FROM`.

# Changelog

{VERSION_NOTES}
"""
    metadata = {
        "ownerSlug": KAGGLE_OWNER,
        "modelSlug": KAGGLE_MODEL_SLUG,
        "instanceSlug": KAGGLE_INSTANCE_SLUG,
        "framework": KAGGLE_FRAMEWORK,
        "overview": "PyTorch object-detection checkpoints.",
        "usage": usage,
        "licenseName": KAGGLE_MODEL_LICENSE,
        "fineTunable": False,
        "trainingData": [],
        "modelInstanceType": "Unspecified",
        "baseModelInstanceId": 0,
        "externalBaseModelUrl": "",
    }
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / "model-instance-metadata.json"
    path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path


model_metadata_path = write_model_metadata(MODEL_METADATA_DIR)
instance_metadata_path = write_model_instance_metadata(INSTANCE_METADATA_DIR)
print(f"Model metadata written: {model_metadata_path}")
print(f"Model instance metadata written: {instance_metadata_path}")


Model metadata written: /Users/ngocbao/Documents/Document/Semester6/Image Processing/Final/saved_results/kaggle_upload_export/_model_metadata/model-metadata.json
Model instance metadata written: /Users/ngocbao/Documents/Document/Semester6/Image Processing/Final/saved_results/kaggle_upload_export/_instance_metadata/model-instance-metadata.json


In [5]:
def run_command(command: list[str], *, allow_existing_error: bool = False) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(command), flush=True)
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode == 0:
        return result
    combined = f"{result.stdout}\n{result.stderr}".lower()
    if allow_existing_error and any(token in combined for token in ["already", "exist", "409", "duplicate"]):
        print("Resource seems to already exist; continuing.")
        return result
    raise subprocess.CalledProcessError(result.returncode, command, result.stdout, result.stderr)


def upload_model_checkpoints() -> None:
    model_metadata_path = write_model_metadata(MODEL_METADATA_DIR)
    instance_metadata_path = write_model_instance_metadata(INSTANCE_METADATA_DIR)
    print(f"Model metadata written: {model_metadata_path}")
    print(f"Model instance metadata written: {instance_metadata_path}")

    if CREATE_MODEL_IF_MISSING:
        run_command(
            ["kaggle", "models", "create", "-p", str(MODEL_METADATA_DIR)],
            allow_existing_error=True,
        )
    else:
        print("Skipping model creation. Set CREATE_MODEL_IF_MISSING=1 to create it automatically.")

    if CREATE_INSTANCE_IF_MISSING:
        run_command(
            ["kaggle", "models", "instances", "create", "-p", str(INSTANCE_METADATA_DIR), "--dir-mode", "skip"],
            allow_existing_error=True,
        )
    else:
        print("Skipping model instance creation. Set CREATE_INSTANCE_IF_MISSING=1 to create it automatically.")

    run_command([
        "kaggle",
        "models",
        "instances",
        "versions",
        "create",
        KAGGLE_MODEL_INSTANCE_REF,
        "-p",
        str(MODEL_VERSION_DIR),
        "--dir-mode",
        "zip",
        "-n",
        VERSION_NOTES,
    ])


def upload_public_dataset() -> None:
    if not KAGGLE_DATASET_SLUG:
        raise ValueError("Set KAGGLE_DATASET_SLUG=owner/dataset-name before uploading public dataset.")

    command_version = [
        "kaggle",
        "datasets",
        "version",
        "-p",
        str(DATASET_EXPORT_DIR),
        "--dir-mode",
        "zip",
        "-m",
        DATASET_VERSION_NOTES,
    ]
    command_create = [
        "kaggle",
        "datasets",
        "create",
        "-p",
        str(DATASET_EXPORT_DIR),
        "--dir-mode",
        "zip",
    ]

    try:
        run_command(command_version)
    except subprocess.CalledProcessError:
        print("Dataset version upload failed, trying dataset create...")
        run_command(command_create)


if not shutil.which("kaggle"):
    raise RuntimeError("kaggle CLI is not installed. Run: pip install kaggle")

if UPLOAD_TARGET in {"model", "both"}:
    upload_model_checkpoints()
else:
    print("Skipping Kaggle Model upload.")

if UPLOAD_TARGET in {"dataset", "both"}:
    upload_public_dataset()
else:
    print("Skipping Kaggle Dataset upload.")


Skipping Kaggle Model upload.
$ kaggle datasets version -p /Users/ngocbao/Documents/Document/Semester6/Image Processing/Final/saved_results/kaggle_upload_export/public_dataset --dir-mode zip -m Update public object-detection dataset


Starting upload for file public.zip
Upload successful: public.zip (998MB)


  0%|                                               | 0.00/998M [00:00<?, ?B/s]
  0%|                                   | 16.0k/998M [00:00<14:56:12, 19.5kB/s]
  0%|                                      | 144k/998M [00:01<1:59:48, 146kB/s]
  0%|                                      | 176k/998M [00:01<2:15:50, 128kB/s]
  0%|                                      | 192k/998M [00:01<2:38:24, 110kB/s]
  0%|                                      | 368k/998M [00:02<1:18:38, 222kB/s]
  0%|                                      | 512k/998M [00:02<1:01:42, 282kB/s]
  0%|                                        | 880k/998M [00:02<29:28, 591kB/s]
  0%|                                       | 1.19M/998M [00:03<30:24, 573kB/s]
  0%|                                      | 2.12M/998M [00:03<15:27, 1.13MB/s]
  0%|                                      | 2.59M/998M [00:04<13:20, 1.30MB/s]
  0%|▏                                     |

Starting upload for file public.zip
Upload successful: public.zip (998MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/ngocbaotrinhtuan/object-detection-public


  0%|                                               | 0.00/998M [00:00<?, ?B/s]
  0%|                                     | 16.0k/998M [00:00<2:12:50, 131kB/s]
  0%|                                        | 160k/998M [00:00<25:53, 674kB/s]
  0%|                                       | 352k/998M [00:00<15:39, 1.11MB/s]
  0%|                                       | 576k/998M [00:00<14:26, 1.21MB/s]
  0%|                                        | 704k/998M [00:00<26:45, 651kB/s]
  0%|                                        | 800k/998M [00:01<25:51, 674kB/s]
  0%|                                        | 912k/998M [00:01<23:05, 754kB/s]
  0%|                                      | 1.39M/998M [00:01<10:22, 1.68MB/s]
  0%|                                      | 2.86M/998M [00:01<03:4

## Commands

### Upload checkpoint vào Kaggle Models

```bash
UPLOAD_TARGET=model \
RUN_NAMES=retina-resnet101-fpn,retina-baseline \
KAGGLE_OWNER=ngocbaotrinhtuan \
KAGGLE_MODEL_SLUG=object-detection-checkpoints \
KAGGLE_INSTANCE_SLUG=checkpoints \
jupyter nbconvert --to notebook --execute upload.ipynb --inplace
```

### Upload thư mục `public/` vào Kaggle Datasets

```bash
UPLOAD_TARGET=dataset \
PUBLIC_DIR=./public \
KAGGLE_DATASET_SLUG=ngocbaotrinhtuan/object-detection-public \
KAGGLE_DATASET_TITLE="Object Detection Public Dataset" \
jupyter nbconvert --to notebook --execute upload.ipynb --inplace
```

### Download từ Kaggle Model

```bash
kaggle models instances versions download \
  ngocbaotrinhtuan/object-detection-checkpoints/PyTorch/checkpoints/1 \
  -p ./saved_results/.kaggle_model_download \
  --untar \
  --force
```

Format Kaggle Model version:

```text
<owner>/<model-slug>/<framework>/<instance-slug>/<version-number>
```

### Download từ Kaggle Dataset

```bash
kaggle datasets download \
  -d ngocbaotrinhtuan/object-detection-public \
  -p ./.kaggle_download \
  --unzip
```
